# ThemeDrift — Main Pipeline
**COS 484 Final Project · Princeton University · April 2026**

Clusters S&P 500 companies by the language in their 10-K filings and tracks how 
those clusters shift across 2019 → 2021 → 2023. Tests whether NLP-derived themes 
can anticipate thematic ETF compositions before the market formally labels them.

---

## Full Pipeline (across all notebooks and scripts)
- **Step 1 — Data Pull** → `data_pull.ipynb`: Pulls 10-K Item 1 text from SEC EDGAR 
  for 496 S&P 500 companies across 2019, 2021, 2023 (1,461 filings total)
- **Step 2 — Embeddings** → `main.ipynb`: Converts filing text to vectors via TF-IDF, SBERT, E5
- **Step 3 — Clustering** → `main.ipynb`: UMAP + HDBSCAN to discover investment themes
- **Step 4 — Theme Inspection** → `main.ipynb`: Human-readable cluster printout
- **Step 5 — Validation** → `main.ipynb`: Cluster quality against 9 thematic ETF baskets
- **Step 6 — Visualization** → `main.ipynb`: UMAP grids, drift tracks, ETF heatmaps
- **Step 7 — Report** → `report.ipynb`: Full written analysis with figures and conclusions

## This Notebook Covers Steps 2–6
---

In [1]:
from embeddings import run as embed
embed(force=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/92 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/92 [00:00<?, ?it/s]


Embeddings check
  TF-IDF    1461 docs x 100 dims
  SBERT     1461 docs x 384 dims
  E5        1461 docs x 768 dims

  Companies: 496
  Years:     [2019, 2021, 2023]



(     ticker  year                                               text  \
 0         A  2019  Item 1 Business 3 Item 1A Risk Factors 16 Item...   
 1         A  2021  Item 1 Business 3 Item 1A Risk Factors 17 Item...   
 2         A  2023  Item 1 Business 3 Item 1A Risk Factors 17 Item...   
 3      AAPL  2019  Item 1. Business 1 Item 1A. Risk Factors 5 Ite...   
 4      AAPL  2021  Item 1. Business 1 Item 1A. Risk Factors 5 Ite...   
 ...     ...   ...                                                ...   
 1456   ZBRA  2021  Item 1. Business 4 Item 1A. Risk Factors 13 It...   
 1457   ZBRA  2023  Item 1. Business 4 Item 1A. Risk Factors 13 It...   
 1458    ZTS  2019  Item 1. Business Overview 1 Operating Segments...   
 1459    ZTS  2021  Item 1. Business Overview 1 Operating Segments...   
 1460    ZTS  2023  Item 1. Business Overview 1 Operating Segments...   
 
                                              text_clean  
 0     item business item a risk factors item b unres...  
 1  

___
Before we can cluster companies into themes, we need to convert the raw 10-K text into numbers. This step does that using three different embedding methods, which we compare in the clustering step to see which produces the most meaningful themes. We clean the text by stripping numbers, punctuation and extra whitespace, since we care about the language a company uses to describe its business rather than financial figures.

TF-IDF is the simple baseline — it scores words by how distinctive they are to a given document compared to the rest, compressed to 100 dimensions via SVD. SBERT uses a transformer trained for semantic similarity (all-MiniLM-L6-v2), producing 384-dim vectors. E5 is the strongest of the three, built specifically for similarity tasks, outputting 768 dimensions per document.

- Output: 1,461 documents x 3 embedding methods (TF-IDF 100 dims, SBERT 384 dims, E5 768 dims)
- Universe: 496 companies x 3 snapshot years (2019, 2021, 2023)

____

In [2]:
import importlib, clustering
importlib.reload(clustering)
from clustering import run

results_c, summary_c = run(constrained=True, force=True)
results_f, summary_f = run(constrained=False, force=True)


Clustering check  [constrained]
  tfidf  2019  ->  22 themes,  89 noise
  tfidf  2021  ->  24 themes,  121 noise
  tfidf  2023  ->  23 themes,  102 noise
  sbert  2019  ->  15 themes,  217 noise
  sbert  2021  ->  21 themes,  193 noise
  sbert  2023  ->  20 themes,  195 noise
  e5     2019  ->  16 themes,  131 noise
  e5     2021  ->  17 themes,  158 noise
  e5     2023  ->  15 themes,  154 noise


Clustering check  [free]
  tfidf  2019  ->  22 themes,  88 noise
  tfidf  2021  ->  23 themes,  102 noise
  tfidf  2023  ->  22 themes,  105 noise
  sbert  2019  ->  15 themes,  217 noise
  sbert  2021  ->  21 themes,  193 noise
  sbert  2023  ->  20 themes,  195 noise
  e5     2019  ->  16 themes,  131 noise
  e5     2021  ->  17 themes,  158 noise
  e5     2023  ->  15 themes,  154 noise



____

With embeddings in place, we group companies into themes by clustering their vectors. We run HDBSCAN after UMAP dimensionality reduction, independently for each method and each year, giving 9 sets of results to compare. Both constrained (boilerplate-filtered) and free (unfiltered) modes are run for comparison.

Output — themes discovered per method (constrained mode, full S&P 500):
- TF-IDF: 22-24 themes per year, cleanly isolates financials, energy, healthcare, defense, cloud/software, semiconductors, retail, and real estate
- SBERT: 15-21 themes per year, broader clusters, higher noise rate
- E5: 15-17 themes per year, strong semantic separation especially for healthcare and energy
____



In [5]:
exec(open("print_themes.py").read())               # constrained (default)


Theme buckets  [constrained]
Loaded 1461 filings


  TFIDF 2019  |  22 themes  |  89 noise  |  [constrained]

  noise : A, AAPL, ADSK, ALB, ALLE, AMD, AME, AOS, APD, AXON, AXP, BALL, BAX, BKR, BLDR, CCI, CDW, CF, CMG, CRL, CTAS, CTVA, DD, DOV, DOW, DVA, EBAY, ECL, EME, EMR, FE, GOOG, HON, HSIC, HWM, IDXX, IEX, IFF, INTU, IQV, IRM, JBL, JCI, JKHY, KDP, LII, LIN, MAS, MCK, MDLZ, META, MLM, MMM, MNST, MO, MOS, MRNA, MTD, NUE, PG, PKG, PM, PNR, POOL, PWR, ROK, ROL, ROP, RVTY, SBUX, SNA, STE, STLD, TAP, TDY, TECH, TER, TMO, TRMB, TSLA, TSN, VMC, VRT, WMT, WST, XYL, XYZ, ZBH, ZBRA

  theme 0 (25 companies)
  companies : AMAT, AVY, C, CAH, CHD, COP, CTRA, DTE, DVN, ED, EIX, ETR, FCX, GNRC, GWW, KMI, LYB, MS, O, OXY, PHM, PSX, VLO, WAT, WY
  top words : gaap, srt, member, member gaap, fairvalueinputslevel, gaap fairvalueinputslevel, fairvalueinputslevel member, gaap seniornotesmember, seniornotesmember, country

  theme 1 (22 companies)
  companies : ALL, AMGN, APH, BMY, CPRT, DGX, GS, IP, LH


  theme 6 (25 companies)
  companies : AMT, BXP, CBRE, CPT, CSGP, DHI, DOC, EQR, ESS, FRT, KIM, LEN, MCHP, MGM, MU, NDAQ, NVR, PLD, REG, SPG, UDR, VICI, VTR, VTRS, WELL
  top words : real estate, estate, real, tenants, notes, lease, properties, property, homes, reit

  theme 7 (8 companies)
  companies : FDX, FFIV, GEN, LHX, PYPL, SATS, TPL, V
  top words : expressed implied, implied, expressed, materially expressed, equity, certain, market, differ materially, events, condition

  theme 8 (8 companies)
  companies : AIZ, ARES, BX, HLT, INVH, LYV, TEL, TTD
  top words : words, section amended, amended, section, negative, amended section, use words, predicts, performance, potential

  theme 9 (14 companies)
  companies : AWK, BF-B, DAL, EL, EPAM, FDS, FIX, GE, HSY, INTC, KO, MCD, NXPI, PEP
  top words : certain, market, equity, condition, officers, compensation, ii, discussion, schedules summary, exhibits

  theme 10 (24 companies)
  companies : AEE, AEP, AES, ATO, CEG, CMS, CNP, D, DUK


  theme 14 (16 companies)
  companies : ADSK, CDNS, COHR, DELL, EMR, JBL, KEYS, LRCX, MPWR, ROP, SNPS, SWKS, TER, TRMB, TXN, ZBRA
  top words : software, design, solutions, products, customers, manufacturing, semiconductor, test, product, devices

  theme 15 (27 companies)
  companies : ALLE, AME, APD, BKR, BLDR, CARR, CAT, CRH, DE, DOV, EME, GNRC, HAL, HUBB, HWM, IEX, ITW, JCI, LII, LIN, MTD, NUE, OTIS, PCAR, PH, SLB, TDY
  top words : products, solutions, equipment, industrial, systems, customers, energy, services, segment, sales

  theme 16 (18 companies)
  companies : AAPL, AMZN, CHTR, CMCSA, DIS, EA, FOX, FOXA, NFLX, NWS, NWSA, SATS, T, TMUS, TTWO, VRSN, VZ, WBD
  top words : services, content, wireless, fox, network, television, news, entertainment, programming, sports

  theme 17 (8 companies)
  companies : AMP, BEN, IVZ, KKR, PRU, RJF, STT, TROW
  top words : investment, clients, funds, asset, products, advisors, institutional, services, kkr, income

  theme 18 (17 companies)



  theme 0 (29 companies)
  companies : AEE, ALL, AMGN, APH, AZO, BMY, CCL, CI, CPRT, DGX, EOG, EXC, FE, GS, IP, JPM, MCO, MSI, NDSN, NEM, NKE, ON, PFE, SCHW, SO, UPS, WEC, WMB, ZTS
  top words : human capital, human, regulation, capital, competition, overview, environmental, capital resources, officers, intellectual property

  theme 1 (13 companies)
  companies : CLX, ETR, FANG, FTNT, GNRC, GOOG, GOOGL, GPN, HLT, LITE, MRNA, REG, TSLA
  top words : google, products, entergy, mrna, services, cloud, alphabet, net, security, solutions

  theme 2 (9 companies)
  companies : CAH, ED, EIX, HON, KMI, O, PSX, VLO, WY
  top words : gaap, member, ed, member gaap, seniornotesmember, gaap seniornotesmember, psx, srt, notesmember, gaap notespayabletobanksmember

  theme 3 (9 companies)
  companies : APA, C, CTRA, FCX, LYB, MAA, MS, OXY, PHM
  top words : gaap, member, member gaap, srt, apa, seniornotesmember, gaap seniornotesmember, gaap fairvalueinputslevel, fairvalueinputslevel, fairvalueinputs


  theme 10 (15 companies)
  companies : ADP, CBRE, CHRW, CTSH, FOX, FOXA, GD, HII, LMT, NWS, NWSA, STX, TRMB, TROW, TYL
  top words : fox, services, clients, news, solutions, aircraft, navy, systems, adp, digital

  theme 11 (10 companies)
  companies : CARR, COHR, EMR, EQIX, GLW, ICE, INTU, SPGI, SWKS, TXN
  top words : products, solutions, optical, customers, data, manufacturing, global, semiconductor, devices, software

  theme 12 (13 companies)
  companies : BAX, BSX, COR, CVS, HCA, HSIC, IQV, JNJ, PODD, RVTY, SYK, UNH, ZBH
  top words : care, products, health, medical, healthcare, health care, medicare, patient, fda, clinical

  theme 13 (10 companies)
  companies : ABT, ADM, GIS, HRL, IFF, MAS, MKC, MRK, TSCO, TSN
  top words : products, food, ingredients, segment, team members, customers, flavor, foods, team, product

  theme 14 (13 companies)
  companies : APO, BLDR, BX, CRWD, CSCO, CVNA, DD, DXCM, EW, INTC, MCD, MPWR, ORCL
  top words : dr, products, market, oracle, cloud, cu


  theme 0 (26 companies)
  companies : AMAT, APA, C, CAG, CAH, CHD, COP, CTRA, DTE, DVN, ED, EIX, FCX, GWW, HON, KMI, LYB, MAA, MS, O, OXY, PHM, PSX, VLO, WAT, WY
  top words : gaap, member, member gaap, srt, gaap seniornotesmember, seniornotesmember, fairvalueinputslevel, fairvalueinputslevel member, gaap fairvalueinputslevel, dte

  theme 1 (30 companies)
  companies : AEE, ALL, AMGN, APH, AZO, BMY, CCL, CI, CPRT, DGX, EOG, EXC, FE, FITB, GS, IP, JPM, MCO, MSI, NDSN, NEM, NKE, ON, PFE, SCHW, SO, UPS, WEC, WMB, ZTS
  top words : human capital, human, regulation, capital, competition, overview, environmental, capital resources, officers, intellectual property

  theme 2 (18 companies)
  companies : BXP, CBRE, CPT, DHI, DLR, EQR, ESS, FRT, INVH, LEN, LOW, NVR, PLD, ROL, SPG, UDR, VTR, WELL
  top words : real estate, estate, homes, properties, property, homebuilding, real, communities, mortgage, land

  theme 3 (15 companies)
  companies : CASY, DPZ, GIS, HRL, IFF, ITW, MKC, MNST, MOS, 


  theme 11 (64 companies)
  companies : A, ADI, ADP, ADSK, ALGN, ANET, APP, BA, BKNG, BKR, CDNS, CIEN, CRM, CRWD, CTSH, DASH, DELL, EBAY, ETN, FIS, FSLR, FTV, GD, GDDY, GEHC, GEN, GOOG, GOOGL, HII, HOOD, IBM, IRM, ISRG, KLAC, LMT, LRCX, LVS, MAR, MGM, MPWR, MTD, NFLX, NOC, NOW, ORCL, PANW, PH, PLTR, QCOM, RL, ROK, RSG, SLB, SNPS, STX, TDY, TER, TRMB, TSLA, TXT, TYL, UAL, UBER, WYNN
  top words : customers, products, solutions, services, ai, software, cloud, systems, data, new

  theme 12 (16 companies)
  companies : ACGL, APO, BRO, CDW, CNC, EXR, FAST, FICO, KO, NTAP, PAYX, PWR, STLD, TEL, TT, WDAY
  top words : changes, tax, fico, services, market, insurance, workday, solutions, certain, customers

  theme 13 (17 companies)
  companies : AFL, ARES, AVB, AWK, BF-B, BG, DOW, EPAM, FFIV, HLT, HSY, LULU, NEE, PEP, PFG, VICI, WM
  top words : market, certain, equity, words, condition, ii, consolidated, schedule, analysis condition, discussion analysis

  theme 14 (22 companies)
  companie

In [3]:
from validation import run
overlap_results, lead_results, summary = run()


Validation Results

Part A - ETF Overlap (best match per cluster, F1 > 0.2 shown)
------------------------------------------------------------

  TFIDF 2019
    cluster 0    FINX_fintech               F1=0.25  P=0.20  R=0.33  overlap: SPGI
    cluster 1    ARKG_genomics              F1=0.25  P=0.17  R=0.50  overlap: ABBV
    cluster 3    ARKK_innovation            F1=0.25  P=0.20  R=0.33  overlap: AMZN
    cluster 5    FINX_fintech               F1=0.40  P=0.29  R=0.67  overlap: MA, V
    cluster 6    IGV_software               F1=0.22  P=0.33  R=0.17  overlap: INTU
    cluster 8    IGV_software               F1=0.80  P=1.00  R=0.67  overlap: ADBE, CRM, MSFT, ORCL
    cluster 10   BOTZ_ai_robotics           F1=0.80  P=0.67  R=1.00  overlap: IBM, NVDA
    cluster 11   ARKK_innovation            F1=0.40  P=0.50  R=0.33  overlap: TSLA

  TFIDF 2021
    cluster 0    FINX_fintech               F1=0.40  P=0.33  R=0.50  overlap: MA, V
    cluster 3    HACK_cybersecurity         F1=0.40  P=0.

In [9]:
from visualize import run as viz
viz(constrained=True, force=True)

### Output
Output - themes discovered per method after applying the constrained mode:

* TF-IDF: 22-24 themes per year, cleanly isolates financials (BAC, JPM, WFC), 
  healthcare (ABBV, JNJ, MRK), energy utilities, oil & gas, defense, cloud/software, 
  semiconductors, real estate, and retail across all years
* SBERT: 15-21 themes per year, broader clusters with higher noise (~40%), 
  healthcare and energy stable
* E5: 15-17 themes per year, cleanest semantic separation, by 2023 clearly identifies 
  AI/cloud (CRM, GOOGL, ORCL, NVDA), payments (MA, V), pharma, and clean energy 
  as distinct themes

Universe expanded to full S&P 500: 496 companies x 3 years = 1,461 filings

---

## Known Caveats, Limitations & Potential Improvements

### 1. ETF Launch Lag (Conservative Bias in Lead Time Measurement)
ETFs don't launch the moment a theme emerges — regulatory filing, seed capital, index 
construction and admin delays mean there's typically a 12–24 month gap between a theme 
becoming investable and an ETF actually launching. This means our measured lead times 
(2–3 years on average) are likely **understated**. The true gap between "language signal" 
and "analyst identification" is shorter than the gap between "language signal" and 
"ETF launch." Our early detection results are conservative, not overstated.

### 2. Data Integrity — No Leakage (Confirmed)
Each year's clustering used **only that year's 10-K text** with no cross-year information. 
ETF validation labels were applied **after** clustering as an external check — never used 
as inputs. The constrained TF-IDF mode rebuilds the vectorizer from scratch per year. 
NVDA's placement in an AI cluster from 2019 filings reflects only what its 2019 10-K said.
This is the most important methodological point for academic credibility.

### 3. XBRL Contamination in TF-IDF Clusters
Some filings have financial statement markup (XBRL tags like `gaap`, `srt`, `member`, 
`fairvalueinputslevel`) bleeding into Item 1 text, creating an artificial cluster across 
all methods and years. These terms should be added to `boilerplate.py` to eliminate the 
garbage cluster. Fix: add to MANUAL stopword list before next run.

### 4. High Noise in SBERT and E5 (Full S&P 500 Run)
With 496 companies, SBERT noise sits at ~40% and E5 at ~26–32%. 
Reducing `MIN_CLUSTER_SIZE` from 8 to 5 and `MIN_SAMPLES` from 3 to 2 in 
`clustering.py` should bring noise down to 10–20% without fragmenting themes.
Recommend re-running after fixing the XBRL contamination first.

### 5. Potential Next Steps
- Fix XBRL boilerplate contamination and re-run clustering
- Tune HDBSCAN parameters for lower noise on full S&P 500
- Expand to full S&P 500 historical snapshots beyond 2019/2021/2023
- Apply FinBERT as a fourth embedding method (domain-adapted financial language model)
- Build a proper held-out evaluation: train on 2019–2021, predict 2023 ETF composition
- Add interactive UMAP visualization (e.g. plotly) for exploring cluster composition
---